In [1]:
# ===== P2 CELL 1: Connect Google Drive =====
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ===== P2 CELL 2: Write and freeze the evaluation protocol (pre-registration) =====
import os, json, hashlib, datetime

BASE = os.environ.get("BETH_BASE", "/content/drive/MyDrive/BETH_Triage")
CONFIG_PATH = os.path.join(BASE, "config.json")
PROTOCOL_PATH = os.path.join(BASE, "07_logs", "evaluation_protocol_v1.md")

with open(CONFIG_PATH) as fh:
    config = json.load(fh)

if "protocol" in config:
    raise SystemExit("STOP: the protocol is already frozen. Do not overwrite it. "
                     "Changes must be recorded as dated amendments in config.json.")

# ---------- machine-readable settings that every later phase will read ----------
PLAN = {
    "version": "v1",
    "primary_label": "evil",
    "secondary_label": "sus",
    "train_split": "train (all 8 hosts, labels never used)",
    "calibration": {
        "final_calibration_set": "all 4 validation hosts (normal data, labels never used)",
        "guarantee_check": "leave-one-validation-host-out: calibrate on 3 hosts, measure false-retention on the 4th",
        "autoencoder_early_stopping": "random 10% of training rows (seeded), never validation or test",
    },
    "alphas": [0.001, 0.005, 0.01, 0.05],
    "primary_alpha": 0.01,
    "prevalences_for_precision_formula": [0.01, 0.05, 0.10],
    "recall_levels_for_reading_cost": [0.5, 0.9],
    "secondary_mix": {"prevalences": [0.01, 0.05, 0.10], "seeds": list(range(10)),
                      "k_values": [100, 1000], "attack_sampling": ["random_rows", "contiguous_time_window"]},
    "bootstrap": {"resamples": 1000, "unit": "process (hostName, processId) cluster", "ci": 0.95,
                  "p_value": "two-sided bootstrap p = 2 x min(share of paired differences <= 0, share >= 0)"},
    "multiple_comparisons": "Holm correction across the primary comparisons",
    "primary_levels": ["event", "process"],
    "normal_definitions": {"normal_all": "evil = 0", "normal_clean": "evil = 0 and sus = 0"},
    "macro_recall": "average over attack processes of the share of each process's evil events kept",
    "models": {
        "random": {"seed": 42},
        "rule_userId_ge_1000": {"score": "1 if userId >= 1000 else 0"},
        "rarity": {"score": "-log((count(processName,eventName in train)+1)/(N_train+V))"},
        "ocsvm": {"kernel": "rbf", "gamma": "scale", "nu": 0.01, "train_subsample": 50000},
        "isolation_forest": {"n_estimators": 200, "max_samples": 256, "max_features": 1.0},
        "autoencoder": {"hidden": [64, 32, 8, 32, 64], "activation": "ReLU", "loss": "MSE",
                        "optimizer": "Adam", "lr": 0.001, "batch_size": 1024,
                        "max_epochs": 30, "early_stopping_patience": 3},
    },
    "feature_sets": {
        "F_main": "binarized BETH-paper features + one-hot eventName + one-hot processName "
                  "(training vocabulary + UNKNOWN) + log training frequency of processName "
                  "(0 for names never seen in training) + stackAddresses length + argsNum",
        "F_paper": "binarized BETH-paper features only (ablation)",
        "F_no_user": "F_main without the userId flag (ablation)",
        "F_raw": "raw numeric columns as in the original proposal (ablation)",
    },
    "tie_breaking": {"evaluation": "seeded random order (seed 42) for every method; AUROC uses the standard tie-averaged definition",
                     "exported_queue": "ties shown in timestamp order for investigators"},
    "model_seeds": {"primary": 42, "repeats": [42, 43, 44, 45, 46],
                    "applies_to": ["isolation_forest", "autoencoder", "ocsvm"]},
    "random_seed": config["random_seed"],
}

stamp = datetime.datetime.now().isoformat(timespec="seconds")
TEXT = f"""# Evaluation protocol {PLAN['version']} - Unsupervised Forensic Evidence Triage (Member 3, J26-CS-308)

Frozen on: {stamp}
Status: written BEFORE any model was trained or any score was computed.
Rule: this file is never edited. Any change is added as a dated amendment in config.json.

## 1. Research question
Can an evidence queue built from behavioural baselines learned only from normal eBPF telemetry
find attack evidence within a fixed investigator review budget better than simple baselines,
while keeping a pre-set, controlled share of normal records out of the queue?

## 2. Hypotheses
H1 (calibration): at alpha = {PLAN['primary_alpha']}, the false-retention rate on normal data from each
   validation host not used for calibration is at most 2 x alpha (leave-one-host-out).
H2 (effectiveness): at alpha = {PLAN['primary_alpha']}, Isolation Forest and the autoencoder retain more
   attack evidence than the random, userId-rule, rarity and One-Class SVM baselines, measured as
   (a) event-level recall on evil and (b) process-level recall (attack processes kept).
H3 (ranking): the proposed models achieve higher AUROC than every baseline, at event level and
   at process level.
Event level and process level are co-primary, because profiling showed that a few processes
contain most attack events, so event-level numbers alone mostly reflect those processes.
H2 and H3 are supported for a comparison only if its Holm-adjusted bootstrap p-value is below 0.05
and the 95% confidence interval of the paired difference excludes zero.
Negative results will be reported in full.

## 3. Data roles (labels are never model inputs)
- Training: all 8 training hosts. Used only to fit preprocessing and models.
- Autoencoder early stopping: a seeded random 10% of training rows.
- Calibration: the 4 validation hosts (normal activity only). Used to turn scores into the
  Investigation Priority Score (IPS) and to set thresholds. No tuning on validation labels.
- Guarantee check: leave-one-validation-host-out (calibrate on 3 hosts, test on the 4th).
- Test: the single test host. Each pre-registered configuration (method x feature set) is
  scored on it after this protocol and all code settings are frozen. No setting is changed
  after test scores have been seen.
- Primary positive label: evil. Secondary: sus. Decisions D1 and D2 in config.json.

## 4. Investigation Priority Score and threshold
For a record x with anomaly score s(x) and calibration scores c_1..c_n:
  p(x)   = (1 + number of c_i >= s(x)) / (n + 1)
  IPS(x) = 1 - p(x)
A record enters the queue when p(x) <= alpha. Primary alpha = {PLAN['primary_alpha']};
sensitivity alphas = {PLAN['alphas']}. Ranking uses the raw score s(x).
Ties: for evaluation, tied records are put in a seeded random order (seed 42) for every method,
so no method gains from the order of the file; AUROC uses the standard tie-averaged definition.
In the exported queue for investigators, tied records are shown in timestamp order.

## 5. Methods compared (settings fixed in advance, no tuning on test data)
{json.dumps(PLAN['models'], indent=2)}

## 6. Feature sets
{json.dumps(PLAN['feature_sets'], indent=2)}
F_main is the primary feature set. The others are ablations.
The log-frequency feature is included because a one-hot column for UNKNOWN is always zero in
training, so tree-based models cannot split on it; the frequency feature lets every model see
how familiar a process name is.

## 7. Primary metrics (unmodified test data only)
1. False-retention rate at each alpha: share of normal records kept
   (a) on each held-out validation host, (b) on the test host's normal records, reported
   for normal_all (evil = 0) and normal_clean (evil = 0 and sus = 0).
2. Recall at each alpha: share of evil records kept, plus macro recall (average over
   attack processes of the share of each process's evil events kept).
3. AUROC on the test host (evil vs not evil).
4. Reading cost: number of normal test records (normal_all) ranked above the point where
   {[int(r*100) for r in PLAN['recall_levels_for_reading_cost']]}% of evil records have been reached.
5. Queue precision at attack shares {[int(p*100) for p in PLAN['prevalences_for_precision_formula']]}%, computed as
   precision = pi*R / (pi*R + (1 - pi)*FRR) from the measured R and FRR (normal_all).
   This assumes attack and normal records keep the same score distributions at other shares.
6. Process level (co-primary): a process is identified by (hostName, processId); its score is the
   maximum event score; it is positive if any of its events is evil. Metrics 1-3 at this level.
   Process-level p-values are calibrated on validation processes (maximum score per process);
   alphas smaller than 1/(number of calibration processes + 1) cannot be reached and are reported as such.
7. Secondary label sus: metrics 1-3 repeated.

## 8. Secondary robustness check
Real-record mixes at {[int(p*100) for p in PLAN['secondary_mix']['prevalences']]}% attack, built from all test-host
evil = 0 records plus sampled evil records (random rows and contiguous time windows),
{len(PLAN['secondary_mix']['seeds'])} seeds each. Reported: Precision@K for K = {PLAN['secondary_mix']['k_values']}, PR-AUC with its no-skill value.
No record is created or modified.

## 9. Uncertainty and statistics
Cluster bootstrap over processes, {PLAN['bootstrap']['resamples']} resamples, 95% percentile intervals.
Paired differences between methods use the same resamples. Bootstrap p-value =
2 x min(share of paired differences <= 0, share of paired differences >= 0).
Holm correction across all primary comparisons: 2 proposed models x 4 baselines x
(recall at alpha = {PLAN['primary_alpha']}, AUROC) x (event, process) = 32 comparisons.

Model randomness: Isolation Forest, the autoencoder and One-Class SVM are trained with seeds
{PLAN['model_seeds']['repeats']}. Headline results use seed {PLAN['model_seeds']['primary']}; the spread across seeds is reported.

## 10. Ablations and sensitivity
Feature-set ablations: F_paper, F_no_user, F_raw. Alpha sensitivity: all alphas listed.
Calibration sensitivity: leave-one-validation-host-out.

## 11. Computational cost
Fit time, scoring time per 10,000 records (requirement NFR-03: under 10 s), peak memory.

## 12. Threats to validity (stated in advance)
One attack on one host; honeypot data from 2021; time-correlated records weaken the
exchangeability assumption behind the guarantee (checked empirically by H1); host names and
timestamps are host-specific and are not used as features; process IDs can be reused by the
operating system, so a (hostName, processId) group may occasionally join two processes.
"""

with open(PROTOCOL_PATH, "w") as fh:
    fh.write(TEXT)
digest = hashlib.sha256(TEXT.encode("utf-8")).hexdigest()

config["protocol"] = {"version": PLAN["version"], "frozen_at": stamp,
                      "file": "07_logs/evaluation_protocol_v1.md", "sha256": digest, "plan": PLAN}
config.setdefault("amendments", [])
with open(CONFIG_PATH, "w") as fh:
    json.dump(config, fh, indent=2)

print("Protocol frozen at", stamp)
print("File   :", PROTOCOL_PATH)
print("SHA-256:", digest)

Protocol frozen at 2026-10-19T05:54:18
File   : /content/drive/MyDrive/BETH_Triage/07_logs/evaluation_protocol_v1.md
SHA-256: d79320a9e6153bcebae0fa02deb24a154687eaaaa85437bfc50e14c99010b35f


In [3]:
# ===== P2 CELL 3: Verify the protocol has not been changed (run any time) =====
import os, json, hashlib

BASE = os.environ.get("BETH_BASE", "/content/drive/MyDrive/BETH_Triage")
with open(os.path.join(BASE, "config.json")) as fh:
    config = json.load(fh)
proto = config["protocol"]
with open(os.path.join(BASE, proto["file"]), "rb") as fh:
    now = hashlib.sha256(fh.read()).hexdigest()
print("Protocol version :", proto["version"])
print("Frozen at        :", proto["frozen_at"])
print("Unchanged        :", now == proto["sha256"])
print("Amendments so far:", len(config.get("amendments", [])))

Protocol version : v1
Frozen at        : 2026-10-19T05:54:18
Unchanged        : True
Amendments so far: 0
